# 01 - Análisis Exploratorio de Datos (EDA)
## Sistema Inteligente de Predicción de Riesgo para Deportistas Amateurs

**Objetivo:** Entender la estructura, calidad y distribución del dataset antes de construir los modelos de Machine Learning.

**Dataset:** `collegiate_athlete_injury_dataset.csv` — 200 atletas universitarios con datos de entrenamiento, recuperación y rendimiento.

## 1. Importaciones y carga del dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

# Carga del dataset
df = pd.read_csv('../data/raw/collegiate_athlete_injury_dataset.csv')
print(f"Dataset cargado correctamente: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

## 2. Información general del dataset

In [ ]:
print("=" * 55)
print("INFORMACIÓN GENERAL")
print("=" * 55)
df.info()

## 3. Estadísticas descriptivas

In [ ]:
df.describe().round(2)

## 4. Análisis de valores nulos

In [ ]:
nulos = df.isnull().sum()
print("Valores nulos por columna:")
print(nulos)
print(f"\nTotal de valores nulos: {nulos.sum()}")
print(f"Porcentaje de completitud: {(1 - nulos.sum() / df.size) * 100:.1f}%")

## 5. Distribución de la variable objetivo (Injury_Indicator)

Este es el punto más crítico del dataset: el **desbalance de clases**. Solo el 7% de los atletas tiene lesión registrada. Si no lo tratamos, el modelo aprenderá a decir "no lesión" siempre y tendrá 93% de accuracy sin aprender nada útil.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

conteo = df['Injury_Indicator'].value_counts()
colores = ['#2ecc71', '#e74c3c']
axes[0].bar(['Sin lesión (0)', 'Con lesión (1)'], conteo.values, color=colores, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución de Injury_Indicator', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de atletas')
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold', fontsize=13)

axes[1].pie(conteo.values, labels=['Sin lesión', 'Con lesión'],
            autopct='%1.1f%%', colors=colores, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción de clases', fontsize=14, fontweight='bold')

plt.suptitle('Desbalance de clases — problema clave del dataset', fontsize=13, style='italic', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/eda_01_desbalance_clases.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAtletas sin lesión: {conteo[0]} ({conteo[0]/len(df)*100:.1f}%)")
print(f"Atletas con lesión: {conteo[1]} ({conteo[1]/len(df)*100:.1f}%)")
print(f"\nRatio de desbalance: {conteo[0]/conteo[1]:.1f}:1")
print("\n→ Solución: aplicar SMOTE en el notebook de features para equilibrar las clases.")

## 6. Distribución de variables numéricas

In [ ]:
cols_numericas = df.select_dtypes(include=np.number).columns.tolist()
cols_numericas = [c for c in cols_numericas if c not in ['Athlete_ID', 'Injury_Indicator']]

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cols_numericas):
    axes[i].hist(df[col], bins=20, color='#3498db', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold', fontsize=11)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')
    media = df[col].mean()
    axes[i].axvline(media, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Media: {media:.1f}')
    axes[i].legend(fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de variables numéricas', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/eda_02_distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Matriz de correlación

In [ ]:
cols_corr = cols_numericas + ['Injury_Indicator']
corr = df[cols_corr].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Matriz de correlación entre variables', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/processed/eda_03_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrelación con Injury_Indicator (ordenada):")
print(corr['Injury_Indicator'].drop('Injury_Indicator').sort_values(key=abs, ascending=False).round(3))

## 8. Análisis bivariado — features vs Injury_Indicator

In [ ]:
features_clave = ['Training_Intensity', 'Training_Hours_Per_Week', 'Recovery_Days_Per_Week',
                  'Fatigue_Score', 'ACL_Risk_Score', 'Load_Balance_Score',
                  'Performance_Score', 'Rest_Between_Events_Days']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
labels = ['Sin lesión', 'Con lesión']
colores = ['#2ecc71', '#e74c3c']

for i, col in enumerate(features_clave):
    data_0 = df[df['Injury_Indicator'] == 0][col]
    data_1 = df[df['Injury_Indicator'] == 1][col]
    axes[i].boxplot([data_0, data_1], labels=labels, patch_artist=True,
                    boxprops=dict(facecolor='lightgray'),
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(axes[i].patches, colores):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].set_ylabel('Valor')

plt.suptitle('Distribución de features según presencia de lesión', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/eda_04_bivariado.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Variables categóricas — Gender y Position

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gender_injury = df.groupby('Gender')['Injury_Indicator'].mean() * 100
axes[0].bar(gender_injury.index, gender_injury.values, color=['#9b59b6', '#3498db'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Tasa de lesión por género (%)', fontweight='bold')
axes[0].set_ylabel('% con lesión')
for i, v in enumerate(gender_injury.values):
    axes[0].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontweight='bold')

position_injury = df.groupby('Position')['Injury_Indicator'].mean() * 100
position_injury = position_injury.sort_values(ascending=False)
axes[1].bar(position_injury.index, position_injury.values, color='#e67e22', edgecolor='white', linewidth=1.5)
axes[1].set_title('Tasa de lesión por posición (%)', fontweight='bold')
axes[1].set_ylabel('% con lesión')
for i, v in enumerate(position_injury.values):
    axes[1].text(i, v + 0.2, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('../data/processed/eda_05_categoricas.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Conclusiones del EDA

In [ ]:
print("=" * 60)
print("CONCLUSIONES DEL EDA")
print("=" * 60)

print("""
1. DATASET LIMPIO
   - 200 registros, 17 columnas, 0 valores nulos.
   - Listo para procesamiento sin imputación necesaria.

2. DESBALANCE DE CLASES (problema crítico)
   - 93% sin lesión vs 7% con lesión (ratio 13:1).
   - Acción: aplicar SMOTE en el notebook de features.

3. VARIABLES MÁS RELEVANTES (por correlación con Injury_Indicator)
   - ACL_Risk_Score: correlación directa con el target.
   - Fatigue_Score y Training_Intensity: más altos en lesionados.
   - Recovery_Days_Per_Week y Load_Balance_Score: más bajos en lesionados.

4. FEATURES A GENERAR EN NOTEBOOK 02
   - Horas de sueño (correlacionadas inversamente con Fatigue_Score).
   - Calidad de sueño (correlacionada con Recovery_Days_Per_Week).
   - Déficit de sueño acumulado (derivado de horas de sueño).
   - Comidas por día (correlacionadas con Performance_Score).
   - Hidratación en litros (correlacionada con Training_Hours_Per_Week).

5. SIGUIENTE PASO
   - Notebook 02_Features.ipynb: limpieza, features sintéticas y preparación
     del dataset final para entrenar los modelos.
""")